# 03 ACID and the Delta Log

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Open the <code>_delta_log</code> and read it: commit files, the actions inside them (<code>add</code>, <code>remove</code>, <code>metaData</code>, <code>protocol</code>, <code>commitInfo</code>), per-file statistics, and checkpoints. Then connect what you see to ACID: atomic commits, isolation by snapshots, and optimistic concurrency control.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
"How does Delta provide ACID guarantees?" is one of the core spoken answers in this learning path. Seeing the actual files makes the explanation concrete, and it explains real behaviour: why a failed job leaves no partial data, why two jobs writing to one table can conflict, and why reading a big table starts with reading the log.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 ACID in Delta</b><br>
| Property | How Delta provides it |<br>|---|---|<br>| <b>Atomicity</b> | A change becomes visible only when its commit file <code>N.json</code> is created. Creating that file is all-or-nothing |<br>| <b>Consistency</b> | Schema enforcement and constraints are checked before committing |<br>| <b>Isolation</b> | Readers use a <b>snapshot</b> (a specific version) and never see uncommitted files. Writers use <b>optimistic concurrency control</b> |<br>| <b>Durability</b> | Commits and data files are stored in durable cloud storage |<br><br>
<b>Optimistic concurrency control (OCC)</b>: a writer reads the latest version, writes its data files, then tries to commit version N+1. If another writer already committed N+1, Delta checks whether the two changes <b>conflict</b> (for example, both touched the same files). If not, it retries as N+2. If they do, it fails with a concurrent modification error.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
Two jobs write to the same <code>events</code> table: a streaming job appends every minute, and a nightly job deletes rows older than 2 years. Appends and deletes of old data touch different files, so OCC lets both commit safely. When a second nightly job was added that <b>updated</b> recent rows at the same time as another update, they sometimes collided with <code>ConcurrentUpdateException</code>. The fix was to schedule them in sequence, or make them work on different partitions.
</div>

## Setup: a path-based table you can open

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a Delta table at a volume path so its log files are visible, and defines helpers to list and read the log.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Managed tables hide their storage, so this lesson uses a Delta table at a volume path, only to inspect the format.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The path of the new table.
</div>

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.delta_lake")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.delta_lake.files")
path = "/Volumes/workspace/delta_lake/files/lesson03/orders"

In [0]:
import os, json
from pyspark.sql import functions as F
from delta.tables import DeltaTable

def log_files():
    return sorted(f for f in os.listdir(f"{path}/_delta_log") if not f.startswith((".", "_staged")))

def latest_version():
    return spark.sql(f"DESCRIBE HISTORY delta.`{path}` LIMIT 1").first()["version"]

def read_commit(version):
    """Return the actions in one commit file as Python dicts."""
    with open(f"{path}/_delta_log/{version:020d}.json") as f:
        return [json.loads(line) for line in f]

orders = spark.createDataFrame(
    [(1, "C1", 120.0), (2, "C2", 80.0), (3, "C1", 45.0), (4, "C3", 300.0)],
    "order_id INT, customer_id STRING, amount DOUBLE",
)
orders.coalesce(2).write.format("delta").mode("overwrite").save(path)
print("table at", path)

## 1. The first commit: protocol, metadata and files

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the log folder and prints the actions in commit 0.<br><br>
<b>Why it matters</b><br>Commit 0 contains everything needed to read version 0: the <b>protocol</b> (minimum reader and writer versions), the <b>metaData</b> (schema, partition columns, properties), one <b>add</b> action per data file, and <b>commitInfo</b> (operation, user, metrics). Each line of the file is one action.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>00000000000000000000.json</code> (often with a <code>.crc</code> checksum file next to it), then actions of type <code>commitInfo</code>, <code>metaData</code>, <code>protocol</code> and two <code>add</code> actions (one per data file).
</div>

In [0]:
print(log_files())
for action in read_commit(0):
    kind = next(iter(action))
    detail = action[kind]
    if kind == "add":
        print(f"add       path={detail['path'][:40]}...  size={detail['size']}")
    elif kind == "metaData":
        print(f"metaData  schema={json.loads(detail['schemaString'])['fields'][0]['name']}, ... partitionColumns={detail['partitionColumns']}")
    elif kind == "protocol":
        print(f"protocol  {detail}")
    elif kind == "commitInfo":
        print(f"commitInfo operation={detail['operation']}")

## 2. File statistics live in the log

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the <code>stats</code> field of each <code>add</code> action: row count and min/max values per column.<br><br>
<b>Why it matters</b><br>These statistics let Delta <b>skip files</b> without opening them: a query for <code>amount &gt; 500</code> can skip any file whose <code>maxValues.amount</code> is below 500. This is data skipping, and it's why clustering data well matters (lesson 10).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
For each data file, <code>numRecords</code> and the min and max of <code>order_id</code>, <code>customer_id</code> and <code>amount</code>.
</div>

In [0]:
for action in read_commit(0):
    if "add" in action:
        stats = json.loads(action["add"]["stats"])
        print(f"records={stats['numRecords']}  min={stats['minValues']}  max={stats['maxValues']}")

## 3. An update: remove the old file, add a new one

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Updates one order and reads commit 1.<br><br>
<b>Why it matters</b><br>Parquet files are immutable, so Delta can't change a row in place. An <code>UPDATE</code> (without deletion vectors) <b>rewrites</b> the files that contain matching rows: it <b>removes</b> the old file and <b>adds</b> a new one with the change. The old file stays on storage, which is what makes time travel possible.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Commit 1 with <code>commitInfo</code> operation <code>UPDATE</code>, one <code>remove</code> action for the old file and one <code>add</code> action for its rewritten replacement. (If deletion vectors are enabled, you may see an <code>add</code> with a <code>deletionVector</code> field instead.)
</div>

In [0]:
spark.sql(f"UPDATE delta.`{path}` SET amount = 85.0 WHERE order_id = 2")
for action in read_commit(1):
    kind = next(iter(action))
    if kind in ("add", "remove"):
        print(f"{kind:<7} {action[kind]['path'][:50]}...")
    elif kind == "commitInfo":
        print(f"commitInfo operation={action[kind]['operation']} metrics={action[kind].get('operationMetrics')}")

## 4. Snapshot isolation: each version is a list of files

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Computes the set of live files at each version by replaying the log, and compares it with what Delta reports.<br><br>
<b>Why it matters</b><br>This is exactly what a reader does: start from a checkpoint or commit 0, apply every <code>add</code> and <code>remove</code> in order, and use the resulting file list as the snapshot. A reader working on version 1 is unaffected by commits that happen afterwards.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Version 0: 2 files. Version 1: still 2 files, but one of them is different. The replay matches the file count from <code>DESCRIBE DETAIL</code>.
</div>

In [0]:
def live_files(up_to_version):
    files = set()
    for v in range(up_to_version + 1):
        for action in read_commit(v):
            if "add" in action:
                files.add(action["add"]["path"])
            elif "remove" in action:
                files.discard(action["remove"]["path"])
    return files

v0, v1 = live_files(0), live_files(1)
print("version 0 files:", len(v0), "| version 1 files:", len(v1), "| files that changed:", len(v0 - v1))
print("DESCRIBE DETAIL numFiles:", spark.sql(f"DESCRIBE DETAIL delta.`{path}`").first()["numFiles"])

## 5. Atomicity: a failed write leaves no trace

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Tries to append rows that fail during the write (a cast error under ANSI mode), then checks the version and row count.<br><br>
<b>Why it matters</b><br>The job may have written some data files before failing, but it never created the commit file, so those files are invisible: readers see the table exactly as before. No cleanup is needed (<code>VACUUM</code> later removes the orphan files).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The append fails. The latest version is still 1 and the row count is still 4.
</div>

In [0]:
bad = spark.createDataFrame([(5, "C9", "not a number")], "order_id INT, customer_id STRING, amount STRING") \
    .withColumn("amount", F.col("amount").cast("double"))
try:
    bad.write.format("delta").mode("append").save(path)
except Exception as e:
    print("write failed:", type(e).__name__)

print("latest version:", latest_version(), "| rows:", spark.read.format("delta").load(path).count())

## 6. Checkpoints

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Makes 12 small commits and lists the log again.<br><br>
<b>Why it matters</b><br>Replaying thousands of JSON files would make every read slow. Every 10 commits (by default), Delta writes a <b>checkpoint</b>: a Parquet file with the full table state at that version, plus <code>_last_checkpoint</code> pointing to it. Readers start from the latest checkpoint and apply only the later JSON commits.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
JSON files up to version 13 or so, a <code>...00010.checkpoint.parquet</code> file, and a <code>_last_checkpoint</code> file.
</div>

In [0]:
for i in range(12):
    spark.createDataFrame([(100 + i, "C5", float(i))], orders.schema).write.format("delta").mode("append").save(path)

files = log_files()
print("json commits :", len([f for f in files if f.endswith(".json")]))
print("checkpoints  :", [f for f in files if "checkpoint" in f])

## 7. Optimistic concurrency in practice

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Explains the conflict rules and shows the isolation level of the table.<br><br>
<b>Why it matters</b><br>Most concurrent operations succeed: blind appends never conflict with each other, and operations touching different files (or partitions) don't conflict. Conflicts happen when two transactions <b>read and modify the same files</b>, for example two <code>UPDATE</code>s or <code>MERGE</code>s on overlapping data.
</div>

| Operation A \ B | Append | Delete / Update / Merge on other files | On the same files |
|---|---|---|---|
| Append | ✅ | ✅ | ✅ |
| Delete / Update / Merge | ✅ | ✅ (different files or partitions) | ❌ `ConcurrentModificationException` family |
| `OPTIMIZE` | ✅ | ✅ | ❌ can conflict with updates and deletes |

Common error classes: `ConcurrentAppendException`, `ConcurrentDeleteReadException`, `ConcurrentDeleteDeleteException`, `MetadataChangedException`, `ProtocolChangedException`.

Isolation levels: **WriteSerializable** (default, allows more concurrency) and **Serializable** (strictest). Row-level concurrency on Databricks (with deletion vectors) reduces conflicts further.

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The latest commits with their recorded isolation level and whether they were blind appends. <code>WriteSerializable</code> is the default isolation level for Delta tables. Blind appends, which read nothing from the table, conflict with nothing, so they are recorded as <code>Serializable</code>.
</div>

In [0]:
display(spark.sql(f"DESCRIBE HISTORY delta.`{path}` LIMIT 3").select("version", "operation", "isolationLevel", "isBlindAppend"))

## Under the hood

```
Writer (version 13 is latest)
 1. read snapshot 13                                        Readers keep using their snapshot
 2. write new Parquet files                                 (version 13) until they refresh:
 3. try to create _delta_log/00000000000000000014.json  ◀── never see uncommitted files
      ├─ created: commit succeeded, version 14 is visible
      └─ already exists (another writer won):
           read commit 14, check for logical conflicts
             ├─ no conflict: retry as version 15
             └─ conflict: fail with ConcurrentModificationException
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> each successful operation is one commit. Failed ones leave only orphan files.

**Interview soundbite:** "Delta writes data files first and then commits by atomically creating the next log entry. Readers only see committed files, so they get consistent snapshots. Concurrent writers use optimistic concurrency: if the version is taken, Delta checks for conflicts and either retries or fails, which keeps the table consistent without locks."

In [0]:
spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select("version", "operation", "operationMetrics").show(5, truncate=60)

In [0]:
# Optional cleanup
dbutils.fs.rm("/Volumes/workspace/delta_lake/files/lesson03", True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>ConcurrentAppendException</code> / <code>ConcurrentDeleteReadException</code></b><br>
Two jobs modified overlapping data at the same time. Run them in sequence, make them touch different partitions, add explicit predicates to <code>MERGE</code>/<code>UPDATE</code> conditions, or enable row-level concurrency on Databricks.<br><br>
**⛔ Problem: editing or deleting <code>_delta_log</code> files**<br>Never do it: it corrupts the table. Use <code>RESTORE</code> to go back.<br><br>
<b>⛔ Problem: orphan files after failed jobs</b><br>They're invisible to readers. <code>VACUUM</code> removes them after the retention period.<br><br>
<b>⛔ Problem: very slow reads of a table with millions of commits</b><br>Checkpoints should prevent this. Check the log retention and that checkpoints are being written.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How does Delta Lake provide ACID guarantees?</b><br>
Writers create new Parquet files and then commit by atomically creating the next numbered JSON file in <code>_delta_log</code>, listing the files added and removed. That makes each change atomic. Readers build a snapshot from the log and only see committed files, which gives isolation. Schema checks and constraints run before the commit, for consistency. Cloud storage makes commits durable. Concurrent writers use optimistic concurrency control.<br><br>

<b>🎤 2. What is in a Delta commit file?</b><br>
One JSON action per line: <code>add</code> and <code>remove</code> for data files (with per-file statistics), <code>metaData</code> for the schema and properties, <code>protocol</code> for the required reader and writer versions, and <code>commitInfo</code> with the operation, user and metrics.<br><br>

<b>🎤 3. What is a checkpoint in the Delta log?</b><br>
A Parquet file, written every 10 commits by default, that stores the full table state at that version. Readers start from the latest checkpoint instead of replaying every JSON commit.<br><br>

<b>🎤 4. What is optimistic concurrency control in Delta?</b><br>
Writers don't lock the table. They prepare their changes and try to commit the next version. If another commit got there first, Delta checks whether the changes conflict. Non-conflicting changes, such as appends, are retried automatically, and conflicting ones fail with a concurrent modification exception.<br><br>

<b>🎤 5. What happens if a write fails halfway?</b><br>
Its commit file is never created, so the files it wrote are invisible and the table stays at the previous version. <code>VACUUM</code> later removes the orphan files.<br><br>

<b>🎤 6. How does an <code>UPDATE</code> work if Parquet files are immutable?</b><br>
Delta rewrites the files containing matching rows (copy-on-write), removing the old files and adding new ones in one commit. With deletion vectors, it can instead mark rows as deleted and write only the changed rows.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A job that appends to a Delta table fails after writing some data files. What do readers of the table see?</b><br>
A. The partially written rows<br>
B. The table as of the last successful commit<br>
C. An error until the table is repaired<br>
D. Duplicated rows<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Uncommitted files are never part of a snapshot.</details><br><br>

<b>Q2. Which mechanism prevents two concurrent writers from corrupting a Delta table?</b><br>
A. Table-level locks held for the duration of the job<br>
B. Optimistic concurrency control with atomic commits to the transaction log<br>
C. Writing each job to a separate cluster<br>
D. Spark's DAG scheduler<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Delete the orders of customer <code>C1</code> on the path table and print the actions of the new commit</li><li>Using <code>live_files</code>, show how many data files the table had at version 1 and at the latest version</li><li>Find, from the log, which commit added the most records (sum <code>numRecords</code> of its <code>add</code> actions)</li><li>In your own words, explain why a failed append doesn't need to be cleaned up before the next run</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Delete and inspect the commit
spark.sql(f"DELETE FROM delta.`{path}` WHERE customer_id = 'C1'")
latest = latest_version()
for action in read_commit(latest):
    kind = next(iter(action))
    print(kind, action[kind].get("path", action[kind].get("operation", ""))[:60])

# 2. Live files per version (replay works across checkpoints because all JSON commits are still present)
print("version 1:", len(live_files(1)), "files | latest:", len(live_files(latest)), "files")

# 3. Commit that added the most records
added = {}
for v in range(latest + 1):
    added[v] = sum(json.loads(a["add"]["stats"])["numRecords"] for a in read_commit(v) if "add" in a and a["add"].get("stats"))
best = max(added, key=added.get)
print("version with most added records:", best, "->", added[best])

# 4. The failed write never created its commit file, so its data files aren't part of any snapshot.
#    The next run commits a new version normally, and VACUUM eventually deletes the orphan files.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>_delta_log/N.json</code> = one commit, with actions: <code>add</code>, <code>remove</code>, <code>metaData</code>, <code>protocol</code>, <code>commitInfo</code></li><li><code>add</code> actions carry per-file statistics (row count, min/max), used for data skipping</li><li>Updates rewrite whole files (copy-on-write), or use deletion vectors</li><li>A snapshot = replay of add/remove actions. Readers never see uncommitted files</li><li>Checkpoints every 10 commits speed up reading the log</li><li>Optimistic concurrency: commit N+1 or detect a conflict and retry or fail. Appends rarely conflict</li></ul>
</div>

| Look at | How |
|---|---|
| Log files | `os.listdir(path + "/_delta_log")` (path tables) |
| Commit content | read `N.json` line by line |
| Operations and metrics | `DESCRIBE HISTORY t` |
| Isolation level | `history()` → `isolationLevel` |
| Current files | `DESCRIBE DETAIL t` → `numFiles` |

## Git commit

```
git add 03_delta_lake/03_acid_and_delta_log.ipynb
git commit -m "add 03_03 acid and delta log notebook"
```